In [ ]:
import time
import torch
import pandas as pd
from thop import profile
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
import math
import os
from PIL import Image

import pandas as pd

In [4]:
class DynamicConvNetwork(nn.Module):
    def __init__(self, num_classes=100, K=4, embed_dim=64):
        super().__init__()
        
        # 第一層直接使用靜態卷積（或動態卷積）將3通道轉換為embed_dim通道
        self.first_conv = nn.Conv2d(3, embed_dim, kernel_size=3, padding=1)
        
   
        self.dynamic_conv1 = self._make_dynamic_conv_layer(embed_dim, 64, K)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.dynamic_conv2 = self._make_dynamic_conv_layer(64, 128, K)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.dynamic_conv3 = self._make_dynamic_conv_layer(128, 256, K)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # 分類頭
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )
        
    def _make_dynamic_conv_layer(self, in_channels, out_channels, K):
        # 建立一個容器來放置所有模組和參數
        layer = nn.Module()
        
        # 注意力機制
        layer.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, 64, 1),
            nn.ReLU(),
            nn.Conv2d(64, K, 1),
            nn.Softmax(dim=1)
        )
        
        # 卷積權重和偏置
        layer.weight_tensors = nn.ParameterList([
            nn.Parameter(torch.Tensor(out_channels, in_channels, 3, 3)) 
            for _ in range(K)
        ])
        
        # 初始化權重
        for weight in layer.weight_tensors:
            nn.init.kaiming_uniform_(weight, a=math.sqrt(5))
            
        layer.bias = nn.Parameter(torch.zeros(out_channels))
        
        # 批次歸一化和激活
        layer.bn = nn.BatchNorm2d(out_channels)
        layer.relu = nn.ReLU(inplace=True)
        
        return layer
    
    def _apply_dynamic_conv(self, x, layer):
        batch_size = x.size(0)
        
        # 計算注意力權重
        attn = layer.attention(x)  # [B, K, 1, 1]
        
        # 對於每個樣本執行卷積
        out = torch.zeros(batch_size, layer.weight_tensors[0].size(0), 
                         x.size(2), x.size(3), device=x.device)
        
        for b in range(batch_size):
            # 組合卷積核權重
            kernel_weight = 0
            for k in range(len(layer.weight_tensors)):
                kernel_weight += attn[b, k, 0, 0] * layer.weight_tensors[k]
                
            # 執行卷積
            out[b] = F.conv2d(x[b:b+1], kernel_weight, layer.bias, padding=1)
        
        # 批正規化和激活
        out = layer.bn(out)
        out = layer.relu(out)
        
        return out
    
    def forward(self, x):
        
        # 通道嵌入
        x = self.first_conv(x)
        
        
        x = self._apply_dynamic_conv(x, self.dynamic_conv1)
        x = self.pool1(x)
        
        x = self._apply_dynamic_conv(x, self.dynamic_conv2)
        x = self.pool2(x)
        
        x = self._apply_dynamic_conv(x, self.dynamic_conv3)
        x = self.pool3(x)
        

        x = self.classifier(x)
        
        return x

In [10]:
# === 自定義資料集 ===
class MiniImageNetDataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []
        with open(txt_file, 'r') as f:
            for line in f:
                img_path, label = line.strip().split()
                self.data.append((img_path, int(label)))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        full_path = os.path.join(self.root_dir, img_path)
        img = Image.open(full_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# === 工具函數 ===
def compute_params_flops(model, input_shape=(3, 64, 64)):
    dummy_input = torch.randn(1, *input_shape).to(next(model.parameters()).device)
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    return params, macs * 2

def measure_inference_time(model, input_tensor, repeat=100):
    model.eval()
    with torch.no_grad():
        start = time.time()
        for _ in range(repeat):
            _ = model(input_tensor)
        end = time.time()
    return (end - start) / repeat * 1000

def evaluate_model_accuracy(model, dataset, device, batch_size=64):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


# === 消融實驗主函數 ===
def run_K_ablation_from_checkpoints(K_list, dataset, device, checkpoint_dir, output_csv, input_shape=(3, 64, 64)):
    results = []
    dummy_input = torch.randn(1, *input_shape).to(device)
    for K in K_list:
        print(f"\n==> Evaluating pretrained model for K={K}")
        model = DynamicConvNetwork(K=K).to(device)
        ckpt_path = os.path.join(checkpoint_dir, f"best_DynamicConvNetwork_{K}k.pth")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        
        acc = evaluate_model_accuracy(model, dataset, device)
        params, flops = compute_params_flops(model, input_shape=input_shape)
        inf_time = measure_inference_time(model, dummy_input)
        
        results.append({
            'K': K,
            'Test Accuracy (%)': round(acc * 100, 2),
            'Params': params,
            'FLOPs': flops,
            'Inference Time (ms/img)': round(inf_time, 3)
        })

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\n 結果已儲存至 {output_csv}")
    return df

In [12]:
# === 執行區塊 ===
transform = transforms.Compose([
    transforms.Resize(96),
    transforms.CenterCrop(84),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = MiniImageNetDataset(
    txt_file='dataset/test.txt',
    root_dir='dataset',
    transform=transform
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_list = [2, 4, 8]
checkpoint_dir = './checkpoints'
output_csv_path = os.path.join(checkpoint_dir, 'K_ablation_results.csv')

result_df = run_K_ablation_from_checkpoints(K_list, test_dataset, device, checkpoint_dir, output_csv_path)
display(result_df)


==> Evaluating pretrained model for K=2


/tmp/ipykernel_2413/3105802343.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))



==> Evaluating pretrained model for K=4



==> Evaluating pretrained model for K=8



 結果已儲存至 ./checkpoints/K_ablation_results.csv


,K,Test Accuracy (%),Params,FLOPs,Inference Time (ms/img)
0,2,53.78,45354.0,18665246.0,1.540
1,4,60.89,45744.0,18666050.0,1.697
2,8,57.11,46524.0,18667658.0,2.108
